# TP — Churn de Clientes: Preparación de Datos para el Modelo

Este notebook deja la base **lista para modelar**, partiendo del dataset ya limpio del EDA.
La salida procesada se guarda en `data/processed/` y la consume el notebook de modelado.

**La limpieza de categorías NO se hace acá:** ocurre una sola vez en el EDA
(`01_EDA_Churn.ipynb`), que guarda `data/processed/dataset_limpio.csv`. Acá cargamos ese
archivo ya limpio (no se duplica el paso de limpieza).

**Orden del pipeline (clave para evitar data leakage):**
1. **Split estratificado train/test** (antes de cualquier estadística)
2. Imputación de nulos — mediana calculada **solo en train**
3. Tratamiento de outliers — cap en percentil 99 **fit en train**
4. Feature engineering (row-wise)
5. One-Hot Encoding de categóricas nominales — **fit en train**
6. Guardado de la base procesada (versiones con y sin `Complain`)

La lógica reutilizable vive en `src/preprocessing.py`.

In [1]:
import sys
sys.path.append('..')  # para importar el módulo src/

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from src import preprocessing as prep
from src.preprocessing import RANDOM_STATE, TARGET, ID_COL, CAT_NOMINAL

pd.set_option('display.max_columns', None)
print('✅ Setup completo. Funciones de preparación importadas desde src/preprocessing.py')

✅ Setup completo. Funciones de preparación importadas desde src/preprocessing.py


---
## 1. Carga de la base limpia

Cargamos `data/processed/dataset_limpio.csv`, que produce el EDA tras unificar las categorías
inconsistentes (CC/Credit Card, COD/Cash on Delivery, Phone/Mobile Phone, Mobile/Mobile Phone).
Así la limpieza ocurre **una sola vez** (en el EDA) y no se repite acá.

> Requiere haber corrido `01_EDA_Churn.ipynb` antes, que genera ese archivo.

In [2]:
df = pd.read_csv('../data/processed/dataset_limpio.csv')
print(f'Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas (ya con categorías unificadas)')
print(f'Churn: {df[TARGET].mean()*100:.1f}% positivo')
df.head(3)

Shape: 5,630 filas × 20 columnas (ya con categorías unificadas)
Churn: 16.8% positivo


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Mobile Phone,1,8.0,UPI,Male,3.0,4,Mobile Phone,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Mobile Phone,1,30.0,Debit Card,Male,2.0,4,Mobile Phone,3,Single,6,1,14.0,0.0,1.0,3.0,120


---
## 2. Split estratificado train/test

Separamos **antes** de imputar o calcular cualquier estadística. Así, las medianas y
percentiles se calculan solo con train y el test queda 100% "no visto" (sin leakage).

Parámetros (de `decisions.md`): `test_size=0.2`, `stratify=Churn`, `random_state=42`.

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[TARGET],
    random_state=RANDOM_STATE,
)

print(f'Train: {len(train_df):,} filas | Churn: {train_df[TARGET].mean()*100:.1f}%')
print(f'Test:  {len(test_df):,} filas  | Churn: {test_df[TARGET].mean()*100:.1f}%')
print('✅ Proporción de churn preservada en ambos subsets')

Train: 4,504 filas | Churn: 16.8%
Test:  1,126 filas  | Churn: 16.9%
✅ Proporción de churn preservada en ambos subsets


---
## 3. Imputación de nulos (mediana fit en train)

7 columnas numéricas tienen 4.5–5.5% de nulos. Imputamos con la **mediana** (robusta a
outliers). Las medianas se calculan **solo en train** y se aplican a ambos subsets.

In [4]:
impute_values = prep.fit_impute_values(train_df)
print('Medianas calculadas en TRAIN:')
for col, val in impute_values.items():
    print(f'  {col:30s}: {val:.1f}')

train_df = prep.apply_impute(train_df, impute_values)
test_df = prep.apply_impute(test_df, impute_values)

print(f'\nNulos restantes — train: {train_df.isna().sum().sum()} | test: {test_df.isna().sum().sum()}')

Medianas calculadas en TRAIN:
  Tenure                        : 9.0
  WarehouseToHome               : 14.0
  HourSpendOnApp                : 3.0
  OrderAmountHikeFromlastYear   : 15.0
  CouponUsed                    : 1.0
  OrderCount                    : 2.0
  DaySinceLastOrder             : 3.0

Nulos restantes — train: 0 | test: 0


---
## 4. Tratamiento de outliers (cap en percentil 99)

`Tenure`, `WarehouseToHome` y `NumberOfAddress` tienen valores extremos (probables errores
de carga). En vez de eliminar filas, **cap-eamos** (winsorizamos) al percentil 99 calculado
en train. Esto reduce ruido para modelos lineales sin perder clientes.

In [5]:
caps = prep.fit_outlier_caps(train_df)
print('Cap (percentil 99 de TRAIN):')
for col, cap in caps.items():
    n_cap = (train_df[col] > cap).sum()
    print(f'  {col:18s}: cap={cap:.1f}  (valores cap-eados en train: {n_cap})')

train_df = prep.apply_caps(train_df, caps)
test_df = prep.apply_caps(test_df, caps)
print('\n✅ Outliers cap-eados')

Cap (percentil 99 de TRAIN):
  Tenure            : cap=30.0  (valores cap-eados en train: 41)
  WarehouseToHome   : cap=35.0  (valores cap-eados en train: 44)
  NumberOfAddress   : cap=11.0  (valores cap-eados en train: 3)

✅ Outliers cap-eados


---
## 5. Feature engineering

Variables derivadas de las hipótesis del EDA (todas row-wise, sin leakage):

| Feature | Definición | Hipótesis |
|---|---|---|
| `CashbackPerOrder` | CashbackAmount / OrderCount | H5 (incentivo financiero) |
| `CouponPerOrder` | CouponUsed / OrderCount | intensidad de uso |
| `AppHoursPerDevice` | HourSpendOnApp / NumberOfDeviceRegistered | engagement |
| `IsNewCustomer` | Tenure ≤ 3 meses | H1 (riesgo temprano) |

> Nota: no agregamos `IsSingle` porque el One-Hot de `MaritalStatus` ya lo captura.

In [6]:
train_df = prep.add_features(train_df)
test_df = prep.add_features(test_df)

nuevas = ['CashbackPerOrder', 'CouponPerOrder', 'AppHoursPerDevice', 'IsNewCustomer']
print('Features nuevas (resumen en train):')
print(train_df[nuevas].describe().round(2).T.to_string())

Features nuevas (resumen en train):
                    count   mean    std  min    25%    50%    75%    max
CashbackPerOrder   4504.0  95.95  53.21  0.0  58.33  87.75  129.0  299.0
CouponPerOrder     4504.0   0.64   0.49  0.0   0.33   0.57    1.0    7.0
AppHoursPerDevice  4504.0   0.88   0.45  0.0   0.67   0.75    1.0    3.0
IsNewCustomer      4504.0   0.28   0.45  0.0   0.00   0.00    1.0    1.0


---
## 6. One-Hot Encoding de categóricas nominales

Las nominales no tienen orden natural, así que usamos One-Hot (no LabelEncoder, que impondría
un orden falso). El encoder se **ajusta solo en train** y con `handle_unknown='ignore'` para
no romper si aparece una categoría nueva en test.

In [7]:
y_train = train_df[TARGET].reset_index(drop=True)
y_test = test_df[TARGET].reset_index(drop=True)

X_train_raw = train_df.drop(columns=[ID_COL, TARGET])
X_test_raw = test_df.drop(columns=[ID_COL, TARGET])

num_cols = [c for c in X_train_raw.columns if c not in CAT_NOMINAL]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train_raw[CAT_NOMINAL])
ohe_cols = list(ohe.get_feature_names_out(CAT_NOMINAL))

def transformar(X_raw):
    ohe_arr = ohe.transform(X_raw[CAT_NOMINAL])
    ohe_df = pd.DataFrame(ohe_arr.astype(int), columns=ohe_cols, index=X_raw.index)
    out = pd.concat([X_raw[num_cols], ohe_df], axis=1).reset_index(drop=True)
    return out

X_train = transformar(X_train_raw)
X_test = transformar(X_test_raw)

print(f'Features finales: {X_train.shape[1]} columnas')
print(f'  - numéricas + engineered: {len(num_cols)}')
print(f'  - one-hot: {len(ohe_cols)}  →  {ohe_cols}')
print(f'\nX_train: {X_train.shape} | X_test: {X_test.shape}')

Features finales: 34 columnas
  - numéricas + engineered: 17
  - one-hot: 17  →  ['PreferredLoginDevice_Computer', 'PreferredLoginDevice_Mobile Phone', 'PreferredPaymentMode_Cash on Delivery', 'PreferredPaymentMode_Credit Card', 'PreferredPaymentMode_Debit Card', 'PreferredPaymentMode_E wallet', 'PreferredPaymentMode_UPI', 'Gender_Female', 'Gender_Male', 'PreferedOrderCat_Fashion', 'PreferedOrderCat_Grocery', 'PreferedOrderCat_Laptop & Accessory', 'PreferedOrderCat_Mobile Phone', 'PreferedOrderCat_Others', 'MaritalStatus_Divorced', 'MaritalStatus_Married', 'MaritalStatus_Single']

X_train: (4504, 34) | X_test: (1126, 34)


---
## 7. Versiones con y sin `Complain` + guardado

`Complain` es una señal fuerte pero con **riesgo de leakage** (no sabemos si la queja se
registra antes o después del churn — ver `decisions.md`). Guardamos **dos versiones** de la
base procesada para que el modelado compare el impacto:

- `*_con_complain.csv` — incluye `Complain`
- `*_sin_complain.csv` — la excluye

Cada archivo trae features + la columna `Churn`.

In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Versión con Complain
train_con = X_train.copy(); train_con[TARGET] = y_train.values
test_con = X_test.copy();   test_con[TARGET] = y_test.values

# Versión sin Complain
train_sin = train_con.drop(columns=['Complain'])
test_sin = test_con.drop(columns=['Complain'])

archivos = {
    'train_con_complain.csv': train_con,
    'test_con_complain.csv': test_con,
    'train_sin_complain.csv': train_sin,
    'test_sin_complain.csv': test_sin,
}
for nombre, data in archivos.items():
    ruta = f'../data/processed/{nombre}'
    data.to_csv(ruta, index=False)
    print(f'💾 {nombre:26s} {data.shape}')

print('\n✅ Base procesada guardada en data/processed/')

💾 train_con_complain.csv     (4504, 35)
💾 test_con_complain.csv      (1126, 35)
💾 train_sin_complain.csv     (4504, 34)
💾 test_sin_complain.csv      (1126, 34)

✅ Base procesada guardada en data/processed/


---
## 8. Resumen de la preparación

**Lo que hicimos** (la limpieza de categorías se hace en el EDA, no acá):
1. ✅ Cargamos la base limpia (`dataset_limpio.csv`, output del EDA)
2. ✅ Split estratificado 80/20 **antes** de cualquier estadística (sin leakage)
3. ✅ Imputamos 7 columnas con mediana fit en train
4. ✅ Cap-eamos outliers de 3 columnas al percentil 99 fit en train
5. ✅ Creamos 4 features nuevas a partir de las hipótesis del EDA
6. ✅ One-Hot Encoding de 5 categóricas nominales fit en train
7. ✅ Guardamos 4 CSVs (con/sin `Complain`, train/test)

**Pendiente para el modelado (`03_Modeling_Churn.ipynb`):**
- Baseline (Decision Tree) + Random Forest con `class_weight='balanced'`
- Comparar métricas con y sin `Complain` (validar el riesgo de leakage)
- Métrica principal: **Recall**